# 實戰爬蟲：books.toscrape.com 完整教學

本 Notebook 帶你從零開始，完整走一遍真實爬蟲專案的開發流程：

| 步驟 | 內容 |
|------|------|
| Step 0 | 安裝套件 |
| Step 1 | 分析目標網站（robots.txt + HTML 結構）|
| Step 2 | 取得單頁 HTML（requests）|
| Step 3 | 解析單頁資料（BeautifulSoup）|
| Step 4 | 翻頁爬取全站（50 頁 × 20 本 = 1000 本）|
| Step 5 | 資料清理與分析（pandas）|
| Step 6 | 輸出 CSV |

> **目標網站**：https://books.toscrape.com — 專為練習爬蟲設計的假書店，允許爬取，無需登入。

---
## Step 0：安裝套件

Google Colab 已預裝 `requests` 和 `pandas`，只需安裝 `beautifulsoup4`。

In [ ]:
!pip install beautifulsoup4 --quiet
print("套件安裝完成")

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

print(f"requests  版本：{requests.__version__}")
print(f"pandas    版本：{pd.__version__}")
print("BeautifulSoup 匯入成功")

---
## Step 1：分析目標網站

### 1-A：先看 robots.txt

爬蟲第一步：確認網站是否允許爬取，以及有哪些限制。

In [ ]:
robots = requests.get("https://books.toscrape.com/robots.txt")
print(f"狀態碼：{robots.status_code}")
print("--- robots.txt 內容 ---")
print(robots.text)

> **注意：此網站回傳 404，表示沒有 robots.txt**
> 沒有 robots.txt 不代表可以無限制爬取，但這個網站本身就是為練習爬蟲設計的，官方允許爬取。
> 對真實網站，找不到 robots.txt 時，應保守對待，維持合理的請求間隔。

### 1-B：觀察 URL 規律

在瀏覽器觀察翻頁時 URL 的變化：

```
第 1 頁：https://books.toscrape.com/                          （首頁）
第 2 頁：https://books.toscrape.com/catalogue/page-2.html
第 3 頁：https://books.toscrape.com/catalogue/page-3.html
  ...    ...
第 50 頁：https://books.toscrape.com/catalogue/page-50.html  （最後一頁）
```

規律很清楚：除了第一頁，其餘都是 `catalogue/page-{N}.html`。

### 1-C：用 DevTools 找出每本書的 HTML 結構

在瀏覽器按 `F12` 開啟 DevTools，右鍵點擊任一本書 → 檢查，可以看到：

```html
<article class="product_pod">
  <p class="star-rating Five"></p>          <!-- 評星：第二個 class 值 -->
  <h3>
    <a href="catalogue/.../index.html"
       title="A Light In The Attic">        <!-- 完整書名在 title 屬性 -->
      A Light In ...                         <!-- 文字會被截斷，別用這個 -->
    </a>
  </h3>
  <p class="price_color">£51.77</p>         <!-- 價格 -->
  <p class="availability">In stock</p>      <!-- 庫存 -->
</article>
```

**規劃要抓的欄位：**

| 欄位 | CSS 選擇器 / 屬性 | 備註 |
|------|-------------------|------|
| title | `article.h3.a["title"]` | 取屬性，非文字內容（文字會截斷）|
| price | `.price_color` → 去掉 £ | 轉為 float |
| rating | `p.star-rating` 的第二個 class | 'Five'/'Four'/... |
| availability | `.availability` | strip() 去空白 |
| url | `article.h3.a["href"]` | 相對路徑，需拼接 base URL |

---
## Step 2：取得單頁 HTML

先測試能不能成功取得首頁。

> **關於編碼：** `requests` 有時會自動偵測為 `ISO-8859-1`（即使 HTML 宣告 `UTF-8`），
> 導致 `£` 變成 `Â£`。正確做法是取得回應後手動強制設定 `response.encoding = 'utf-8'`。

In [ ]:
url = "https://books.toscrape.com"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

response = requests.get(url, headers=headers, timeout=10)

print(f"requests 自動偵測編碼：{response.encoding}")  # 可能是 ISO-8859-1（錯誤）

# 強制設定為 UTF-8，避免 £ 變成 Â£
response.encoding = 'utf-8'
print(f"修正後編碼：{response.encoding}")
print(f"狀態碼：{response.status_code}")
print(f"回應長度：{len(response.text):,} 字元")

---
## Step 3：解析單頁資料

### 3-A：建立 BeautifulSoup 物件，熟悉 HTML 結構

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

# 基本確認
print("頁面標題：", soup.title.text.strip())

# 找出所有書籍卡片
articles = soup.select("article.product_pod")
print(f"本頁書籍數量：{len(articles)}")

### 3-B：逐步提取每個欄位

取第一本書，逐欄位拆解確認選擇器正確。

In [ ]:
# 取第一本書做示範
book = articles[0]

# --- 書名 ---
# 注意：a 的文字內容會被截斷，要取 title 屬性才是完整書名
title_text = book.h3.a.text          # 截斷版（錯誤示範）
title_full = book.h3.a["title"]      # 完整書名（正確）
print(f"書名（截斷）：{title_text}")
print(f"書名（完整）：{title_full}")

# --- 價格 ---
# 設定 encoding='utf-8' 後，£ 符號正常，直接替換即可
price_raw = book.select_one(".price_color").text.strip()   # '£51.77'
price_num = float(price_raw.replace("£", ""))              # 51.77
print(f"\n價格（原始）：{price_raw}")
print(f"價格（數字）：{price_num}")

# --- 評星 ---
# p 標籤有兩個 class：['star-rating', 'Five']
rating_classes = book.p["class"]
rating = rating_classes[1]            # 取第二個 class 值
print(f"\n評星 class 列表：{rating_classes}")
print(f"評星：{rating}")

# --- 庫存 ---
stock = book.select_one(".availability").text.strip()
print(f"\n庫存：{stock}")

# --- 連結 ---
# href 已包含 catalogue/，BASE_URL 只需到 https://books.toscrape.com/
href_raw = book.h3.a["href"]          # 'catalogue/a-light-in-the-attic_1000/index.html'
book_url = "https://books.toscrape.com/" + href_raw.replace("../", "")
print(f"\nhref：{href_raw}")
print(f"完整 URL：{book_url}")

### 3-C：封裝成 parse_page 函式

In [ ]:
# href 已包含 catalogue/，base 只需到網域根目錄
BASE_URL = "https://books.toscrape.com/"

def parse_page(soup):
    """從單頁 soup 提取所有書籍資料，返回 List of Dict。"""
    books = []
    for article in soup.select("article.product_pod"):
        books.append({
            "title":        article.h3.a["title"],
            "price":        float(article.select_one(".price_color").text.strip().replace("£", "")),
            "rating":       article.p["class"][1],
            "availability": article.select_one(".availability").text.strip(),
            "url":          BASE_URL + article.h3.a["href"].replace("../", ""),
        })
    return books

# 測試：解析剛才取到的首頁
page1_books = parse_page(soup)
print(f"解析到 {len(page1_books)} 本書")
print()

# 印出前三本
for b in page1_books[:3]:
    print(b)

In [ ]:
# 快速轉成 DataFrame 預覽格式
df_preview = pd.DataFrame(page1_books)
df_preview

---
## Step 4：翻頁爬取全站（50 頁）

### 4-A：封裝 fetch_page 函式（含錯誤處理）

> **重點：** 在 `r.raise_for_status()` 後立即設定 `r.encoding = 'utf-8'`，
> 確保所有頁面的 `£` 符號都能正確解析，不會變成 `Â£`。

In [ ]:
def fetch_page(session, url):
    """發送 GET 請求，返回 BeautifulSoup 物件；失敗返回 None。"""
    try:
        r = session.get(url, timeout=10)
        r.raise_for_status()           # 4xx / 5xx 自動拋出例外
        r.encoding = 'utf-8'           # 強制 UTF-8，避免 £ 變成 Â£
        return BeautifulSoup(r.text, "html.parser")
    except requests.exceptions.HTTPError as e:
        print(f"  HTTP 錯誤 {e.response.status_code}：{url}")
    except requests.exceptions.ConnectionError:
        print(f"  連線失敗：{url}")
    except requests.exceptions.Timeout:
        print(f"  請求逾時：{url}")
    return None


# 測試 fetch_page
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

test_soup = fetch_page(session, "https://books.toscrape.com")
print("fetch_page 測試：", test_soup.title.text.strip() if test_soup else "失敗")

### 4-B：翻頁主控函式 scrape_all

**URL 規律：**
- 第 1 頁：`https://books.toscrape.com`（首頁，URL 不同）
- 第 2～50 頁：`https://books.toscrape.com/catalogue/page-{N}.html`

In [ ]:
def scrape_all(total_pages=50, delay=1.0):
    """
    爬取 books.toscrape.com 指定頁數。

    參數：
        total_pages：要爬幾頁（最多 50）
        delay：每頁之間的等待秒數（預設 1 秒）

    返回：
        包含所有書籍資料的 List of Dict
    """
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})

    all_books = []

    for page in range(1, total_pages + 1):
        # 第一頁 URL 特殊
        if page == 1:
            url = "https://books.toscrape.com"
        else:
            url = f"https://books.toscrape.com/catalogue/page-{page}.html"

        print(f"[{page:2d}/{total_pages}] 爬取：{url}")

        soup = fetch_page(session, url)
        if soup is None:
            print(f"  → 第 {page} 頁失敗，跳過")
            continue

        books = parse_page(soup)
        all_books.extend(books)
        print(f"  → 取得 {len(books)} 本，累計 {len(all_books)} 本")

        time.sleep(delay)   # 遵守爬蟲禮儀

    print(f"\n爬取完成！共 {len(all_books)} 本書")
    return all_books

In [ ]:
# 先測試前 3 頁（60 本），確認一切正常再跑全量
test_books = scrape_all(total_pages=3, delay=1.0)

In [ ]:
# 確認測試結果
df_test = pd.DataFrame(test_books)
print(f"資料形狀：{df_test.shape}")
print(f"欄位：{df_test.columns.tolist()}")
print()
df_test.head(10)

In [ ]:
# 確認 3 頁測試無誤後，執行全量爬取（50 頁，約 50 秒）
# 若只需練習可改為 total_pages=10
all_books = scrape_all(total_pages=50, delay=1.0)

---
## Step 5：資料清理與分析

### 5-A：建立 DataFrame，基本檢查

In [ ]:
df = pd.DataFrame(all_books)

print("=== 資料形狀 ===")
print(df.shape)

print("\n=== 欄位型別 ===")
print(df.dtypes)

print("\n=== 缺失值檢查 ===")
print(df.isnull().sum())

print("\n=== 前五筆 ===")
df.head()

In [ ]:
# 價格統計摘要
print("=== 價格統計 ===")
print(df["price"].describe().round(2))

print(f"\n最貴的書：£{df['price'].max()}")
print(f"最便宜的書：£{df['price'].min()}")
print(f"平均價格：£{df['price'].mean():.2f}")

In [ ]:
# 評星分布
print("=== 各評星數量 ===")
rating_order = ["Five", "Four", "Three", "Two", "One"]
print(df["rating"].value_counts().reindex(rating_order))

### 5-B：資料篩選與排序

In [ ]:
# 五星評分的書，按價格由低到高排列
five_star = df[df["rating"] == "Five"].sort_values("price").reset_index(drop=True)
print(f"五星評分共 {len(five_star)} 本")
print()
five_star[["title", "price"]].head(10)

In [ ]:
# 五星 + 價格低於 15 英鎊的書
best_value = df[(df["rating"] == "Five") & (df["price"] < 15)].sort_values("price")
print(f"五星且低於 £15 的書：共 {len(best_value)} 本")
best_value[["title", "price"]]

In [ ]:
# 各評星等級的平均價格
avg_price_by_rating = (
    df.groupby("rating")["price"]
    .agg(["mean", "count"])
    .round(2)
    .reindex(rating_order)
    .rename(columns={"mean": "平均價格", "count": "書本數量"})
)
print("=== 各評星平均價格 ===")
avg_price_by_rating

In [ ]:
# 最貴的 10 本書
print("=== 最貴的 10 本書 ===")
df.nlargest(10, "price")[["title", "price", "rating"]]

---
## Step 6：輸出 CSV

### 6-A：輸出完整資料

In [ ]:
filename = "books_toscrape_full.csv"

df.to_csv(filename, index=False, encoding="utf-8-sig")
# encoding="utf-8-sig"：讓 Windows Excel 開啟時正確顯示，不會出現亂碼

print(f"已儲存：{filename}")
print(f"總計 {len(df)} 筆資料，{len(df.columns)} 個欄位")

# 重新讀入確認
df_check = pd.read_csv(filename)
print(f"\n讀回驗證：{df_check.shape}")
df_check.head(3)

### 6-B：在 Colab 下載檔案

In [ ]:
# 在 Google Colab 中，執行此儲存格會觸發瀏覽器下載
from google.colab import files
files.download(filename)

### 6-C：輸出篩選後的子集

In [ ]:
# 只輸出五星評分的書
five_star_filename = "books_five_star.csv"
five_star.to_csv(five_star_filename, index=False, encoding="utf-8-sig")
print(f"已儲存：{five_star_filename}（{len(five_star)} 本書）")

---
## 延伸練習

完成基本爬蟲後，可以嘗試以下進階任務：

### 練習 1：爬取每本書的詳情頁

目前我們爬的是書單頁，每本書還有獨立的詳情頁，包含更多資訊（描述、UPC、類型等）。

In [ ]:
# 爬取單本書的詳情頁示範
detail_url = df.iloc[0]["url"]   # 取第一本書的 URL
print(f"詳情頁 URL：{detail_url}")

detail_soup = fetch_page(session, detail_url)

if detail_soup:
    # 書名
    title   = detail_soup.select_one("div.product_main h1").text
    # 描述
    desc_el = detail_soup.select_one("#product_description + p")
    desc    = desc_el.text if desc_el else "無描述"
    # 商品資訊表格
    rows    = detail_soup.select("table.table-striped tr")
    info    = {row.th.text: row.td.text for row in rows}

    print(f"\n書名：{title}")
    print(f"UPC：{info.get('UPC', '—')}")
    print(f"類型：{info.get('Product Type', '—')}")
    print(f"庫存數量：{info.get('Availability', '—')}")
    print(f"\n描述：{desc[:200]}...")

### 練習 2：按分類（Category）爬取

網站左側有 50 個分類（Travel、Mystery、Classics...），每個分類都有獨立的 URL。

In [ ]:
# 取出所有分類名稱與連結
home_soup = fetch_page(session, "https://books.toscrape.com")

categories = []
for li in home_soup.select("ul.nav-list ul li"):
    name = li.a.text.strip()
    href = "https://books.toscrape.com/" + li.a["href"]
    categories.append({"name": name, "url": href})

print(f"共 {len(categories)} 個分類")
for cat in categories[:10]:
    print(f"  {cat['name']:20} → {cat['url']}")

---
## 總結：本次爬蟲的完整程式（可直接複製使用）

In [ ]:
# ============================================================
# books.toscrape.com 完整爬蟲
# 爬取全站 50 頁 1000 本書，輸出 CSV
# ============================================================

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# href 已包含 catalogue/，base 只需到網域根目錄
BASE_URL = "https://books.toscrape.com/"


def fetch_page(session, url):
    """取得網頁，返回 BeautifulSoup；失敗返回 None。"""
    try:
        r = session.get(url, timeout=10)
        r.raise_for_status()
        r.encoding = 'utf-8'   # 強制 UTF-8，避免 £ 變成 Â£
        return BeautifulSoup(r.text, "html.parser")
    except requests.exceptions.RequestException as e:
        print(f"  請求失敗：{e}")
        return None


def parse_page(soup):
    """解析單頁，返回書籍資料的 List of Dict。"""
    books = []
    for article in soup.select("article.product_pod"):
        books.append({
            "title":        article.h3.a["title"],
            "price":        float(article.select_one(".price_color").text.strip().replace("£", "")),
            "rating":       article.p["class"][1],
            "availability": article.select_one(".availability").text.strip(),
            "url":          BASE_URL + article.h3.a["href"].replace("../", ""),
        })
    return books


def scrape_all(total_pages=50, delay=1.0):
    """翻頁爬取全站，返回所有書籍資料。"""
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    all_books = []

    for page in range(1, total_pages + 1):
        url = "https://books.toscrape.com" if page == 1 else \
              f"https://books.toscrape.com/catalogue/page-{page}.html"
        print(f"[{page:2d}/{total_pages}] {url}")

        soup = fetch_page(session, url)
        if soup:
            all_books.extend(parse_page(soup))
        time.sleep(delay)

    print(f"\n完成！共 {len(all_books)} 本書")
    return all_books


# 執行（先跑 3 頁測試；確認無誤後改為 total_pages=50）
books = scrape_all(total_pages=3)

df = pd.DataFrame(books)
df.to_csv("books_toscrape_full.csv", index=False, encoding="utf-8-sig")
print(f"已存為 books_toscrape_full.csv（{df.shape[0]} 筆）")
df.head()